In [ ]:
# Dense Layers – Example 1 – MNIST Handwritten Digits Data

import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize pixel values
x_train, x_test = x_train / 255.0, x_test / 255.0

# Build the model
model = models.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
model.fit(x_train, y_train, epochs=5)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8787 - loss: 0.4363
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9647 - loss: 0.1219
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9770 - loss: 0.0756
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9821 - loss: 0.0564
Epoch 5/5
 868/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9882 - loss: 0.0414

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import notebook_login # connect
import tensorflow as tf

model.save("mnist_model.keras")

In [ ]:
notebook_login()

In [ ]:
# Creating repo
from huggingface_hub import HfApi

# Username/name
repo_id = "amanhusain-123/mnist_model"
api = HfApi()

api.create_repo(repo_id = repo_id,exist_ok = True)

In [ ]:
from huggingface_hub import upload_folder

upload_folder(folder_path="",repo_id=repo_id,repo_type="model")

In [ ]:
from tensorflow import keras
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(repo_id = repo_id,filename = 'mnist_model.keras')
model = keras.models.load_model(model_path)

In [ ]:
# Testing Model
model.predict(x_train)

# `Using Pytorch`

In [ ]:
!pip install torch transformers peft datasets

In [ ]:
from datasets import load_dataset
dataset = load_dataset("SetFit/emotion")

In [ ]:
train_data = dataset['train']
test_data = dataset['test']

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
# Tokenization function
def tokenize_function(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

# Apply tokenization
train_dataset = train_data.map(tokenize_function, batched=True)
test_dataset = test_data.map(tokenize_function, batched=True)

# Convert to PyTorch format
train_dataset.set_format(type="torch",columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch",columns=["input_ids", "attention_mask", "label"])

In [ ]:
from transformers import AutoModelForSequenceClassification

# Load BERT model (6 output classes)
base_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=6)

In [ ]:
base_model

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)

# Define LoRA Configuration
lora_config = LoraConfig(
    r=8,                 # Low-rank adaptation dimension    Adding more rank would be giving more precise accuracy and precision
    lora_alpha=32,       # Scaling factor
    lora_dropout=0.05,   # Dropout rate
    target_modules=["query", "value"]  # Apply LoRA to self-attention layers only
)

# Prepare model for LoRA
base_model = prepare_model_for_kbit_training(base_model)

# Convert model into LoRA-enabled model
peft_model = get_peft_model(base_model, lora_config)

# Print trainable parameters
peft_model.print_trainable_parameters()

Configuration class telling how the training should happen

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./model_checkpoints",   # Where to save model
    num_train_epochs=1,                 # Train for 3 epochs
    per_device_train_batch_size=16,     # 16 samples per GPU/CPU
    eval_strategy="epoch",              # Evaluate after every epoch (using correct parameter name)
    save_strategy="epoch",              # Save model after each epoch
    logging_steps=10,                   # Log training metrics every 10 steps
    load_best_model_at_end=True,         # Automatically load best checkpoint
    fp16=True                            # Use mixed precision for faster training (if GPU supports it)
)

# A high-level class that automates training, evaluation, and saving models.
# It wraps around your model and dataset, handling:
# - Training loops
# - Evaluation during training
# - Model saving & checkpointing

trainer = Trainer(
    model=peft_model,        # LoRA fine-tuned model
    args=training_args,      # Training settings
    train_dataset=train_dataset,  # Training data
    eval_dataset=test_dataset     # Test data
)

trainer.train()